<a href="https://colab.research.google.com/github/Rini43/Asteroid_Diameter_Prediction/blob/main/Exit_exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

## Asteroides Diameter Predictor

# Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Read the data

In [ ]:
# filepath to the .excel file
# Load the dataset from the cloned GitHub repository

filepath = '/content/drive/MyDrive/Data/dataset.csv'
df_asteroid = pd.read_csv(filepath)
df_asteroid.head(20)

## EDA

In [ ]:
# Identifying numerical and categorical data
df_asteroid.info()

In [ ]:
# Identifying dataset size
df_asteroid.shape

In [ ]:
# Statistical Summary
df_asteroid.describe()

## Diameter Distribution Analysis

In [ ]:
# Remove rows with missing diameter values
df_diameter = df_asteroid[df_asteroid['diameter'].notna()].copy()

print(f"Total records: {len(df_asteroid)}")
print(f"Records with diameter data: {len(df_diameter)}")
print(f"Missing diameter values: {df_asteroid['diameter'].isna().sum()}")
print(f"\nDiameter Statistics:")
print(df_diameter['diameter'].describe())

In [ ]:
# Diameter Distribution - Histogram
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Histogram with all data
axes[0, 0].hist(df_diameter['diameter'], bins=100, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Diameter (km)', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Distribution of Asteroid Diameter (All Data)', fontsize=12, fontweight='bold')
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Histogram with log scale (to see smaller asteroids better)
axes[0, 1].hist(df_diameter['diameter'], bins=100, color='lightcoral', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Diameter (km)', fontsize=11)
axes[0, 1].set_ylabel('Frequency (log scale)', fontsize=11)
axes[0, 1].set_title('Distribution of Asteroid Diameter (Log Scale)', fontsize=12, fontweight='bold')
axes[0, 1].set_yscale('log')
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Density plot
df_diameter['diameter'].plot(kind='density', ax=axes[1, 0], color='green', linewidth=2.5)
axes[1, 0].set_xlabel('Diameter (km)', fontsize=11)
axes[1, 0].set_ylabel('Density', fontsize=11)
axes[1, 0].set_title('Density Plot of Asteroid Diameter', fontsize=12, fontweight='bold')
axes[1, 0].grid(alpha=0.3)
axes[1, 0].fill_between(axes[1, 0].get_lines()[0].get_xdata(), 
                        axes[1, 0].get_lines()[0].get_ydata(), alpha=0.3, color='green')

# 4. Box plot with quartile information
box_plot = axes[1, 1].boxplot(df_diameter['diameter'], vert=True, patch_artist=True,
                               boxprops=dict(facecolor='lightblue', alpha=0.7),
                               medianprops=dict(color='red', linewidth=2),
                               flierprops=dict(marker='o', markerfacecolor='red', markersize=5, alpha=0.5))
axes[1, 1].set_ylabel('Diameter (km)', fontsize=11)
axes[1, 1].set_title('Box Plot of Asteroid Diameter', fontsize=12, fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Distribution Insights:")
print(f"  Mean: {df_diameter['diameter'].mean():.2f} km")
print(f"  Median: {df_diameter['diameter'].median():.2f} km")
print(f"  Std Dev: {df_diameter['diameter'].std():.2f} km")
print(f"  Skewness: {df_diameter['diameter'].skew():.2f} (Right-skewed distribution)")
print(f"  Kurtosis: {df_diameter['diameter'].kurtosis():.2f}")

## Key Features for Diameter Prediction

In [ ]:
# Identify key features related to diameter prediction
# Select features with good data availability and physical relevance

key_features = ['H', 'albedo', 'a', 'e', 'i']
print("Key Features Selected for Analysis:")
print(f"  H - Absolute Magnitude")
print(f"  albedo - Reflectivity")
print(f"  a - Semi-major Axis")
print(f"  e - Eccentricity")
print(f"  i - Inclination")

# Check data availability for key features
print("\nData Availability:")
for feature in key_features:
    missing = df_asteroid[feature].isna().sum()
    pct = (missing / len(df_asteroid)) * 100
    print(f"  {feature}: {len(df_asteroid) - missing:,} non-null ({100-pct:.2f}%)")

print(f"\nRecords with all key features and diameter: ", end="")
subset = df_asteroid[key_features + ['diameter']].dropna()
print(f"{len(subset):,} ({(len(subset)/len(df_asteroid)*100):.2f}%)")

## Relationships Between Diameter and Selected Features

In [ ]:
# Create scatter plots showing relationship between diameter and key features
# Using complete cases only

plot_data = df_asteroid[key_features + ['diameter']].dropna()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Relationships Between Asteroid Diameter and Key Features', fontsize=14, fontweight='bold', y=1.00)

# Plot 1: Diameter vs H (Absolute Magnitude)
axes[0, 0].scatter(plot_data['H'], plot_data['diameter'], alpha=0.3, s=10, color='blue')
axes[0, 0].set_xlabel('Absolute Magnitude (H)', fontsize=10)
axes[0, 0].set_ylabel('Diameter (km)', fontsize=10)
axes[0, 0].set_title('Diameter vs Absolute Magnitude', fontsize=11, fontweight='bold')
axes[0, 0].grid(alpha=0.3)
corr_H = plot_data[['H', 'diameter']].corr().iloc[0, 1]
axes[0, 0].text(0.05, 0.95, f'Correlation: {corr_H:.3f}', transform=axes[0, 0].transAxes,
                fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 2: Diameter vs Albedo
axes[0, 1].scatter(plot_data['albedo'], plot_data['diameter'], alpha=0.3, s=10, color='green')
axes[0, 1].set_xlabel('Albedo (Reflectivity)', fontsize=10)
axes[0, 1].set_ylabel('Diameter (km)', fontsize=10)
axes[0, 1].set_title('Diameter vs Albedo', fontsize=11, fontweight='bold')
axes[0, 1].grid(alpha=0.3)
corr_albedo = plot_data[['albedo', 'diameter']].corr().iloc[0, 1]
axes[0, 1].text(0.05, 0.95, f'Correlation: {corr_albedo:.3f}', transform=axes[0, 1].transAxes,
                fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 3: Diameter vs Semi-major Axis (a)
axes[0, 2].scatter(plot_data['a'], plot_data['diameter'], alpha=0.3, s=10, color='red')
axes[0, 2].set_xlabel('Semi-major Axis (a) (AU)', fontsize=10)
axes[0, 2].set_ylabel('Diameter (km)', fontsize=10)
axes[0, 2].set_title('Diameter vs Semi-major Axis', fontsize=11, fontweight='bold')
axes[0, 2].grid(alpha=0.3)
corr_a = plot_data[['a', 'diameter']].corr().iloc[0, 1]
axes[0, 2].text(0.05, 0.95, f'Correlation: {corr_a:.3f}', transform=axes[0, 2].transAxes,
                fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 4: Diameter vs Eccentricity (e)
axes[1, 0].scatter(plot_data['e'], plot_data['diameter'], alpha=0.3, s=10, color='purple')
axes[1, 0].set_xlabel('Eccentricity (e)', fontsize=10)
axes[1, 0].set_ylabel('Diameter (km)', fontsize=10)
axes[1, 0].set_title('Diameter vs Eccentricity', fontsize=11, fontweight='bold')
axes[1, 0].grid(alpha=0.3)
corr_e = plot_data[['e', 'diameter']].corr().iloc[0, 1]
axes[1, 0].text(0.05, 0.95, f'Correlation: {corr_e:.3f}', transform=axes[1, 0].transAxes,
                fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 5: Diameter vs Inclination (i)
axes[1, 1].scatter(plot_data['i'], plot_data['diameter'], alpha=0.3, s=10, color='orange')
axes[1, 1].set_xlabel('Inclination (i) (degrees)', fontsize=10)
axes[1, 1].set_ylabel('Diameter (km)', fontsize=10)
axes[1, 1].set_title('Diameter vs Inclination', fontsize=11, fontweight='bold')
axes[1, 1].grid(alpha=0.3)
corr_i = plot_data[['i', 'diameter']].corr().iloc[0, 1]
axes[1, 1].text(0.05, 0.95, f'Correlation: {corr_i:.3f}', transform=axes[1, 1].transAxes,
                fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 6: Correlation Heatmap
corr_matrix = plot_data[key_features + ['diameter']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0, ax=axes[1, 2],
            cbar_kws={'label': 'Correlation'}, square=True)
axes[1, 2].set_title('Correlation Matrix', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nCorrelation with Diameter (ranked):")
correlations = plot_data[key_features + ['diameter']].corr()['diameter'].drop('diameter').sort_values(ascending=False)
for feature, corr_val in correlations.items():
    print(f"  {feature:10s}: {corr_val:7.4f}")

In [ ]:
# Distribution of key features
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Distributions of Key Predictive Features', fontsize=14, fontweight='bold', y=1.00)

features_to_plot = ['H', 'albedo', 'a', 'e', 'i']
colors = ['blue', 'green', 'red', 'purple', 'orange']

for idx, (feature, color) in enumerate(zip(features_to_plot, colors)):
    row = idx // 3
    col = idx % 3
    
    feature_data = df_asteroid[feature].dropna()
    axes[row, col].hist(feature_data, bins=50, color=color, alpha=0.7, edgecolor='black')
    axes[row, col].set_xlabel(feature, fontsize=10)
    axes[row, col].set_ylabel('Frequency', fontsize=10)
    axes[row, col].set_title(f'Distribution of {feature}', fontsize=11, fontweight='bold')
    axes[row, col].grid(axis='y', alpha=0.3)
    
    # Add statistics
    stats_text = f"Mean: {feature_data.mean():.3f}\nStd: {feature_data.std():.3f}\nMin: {feature_data.min():.3f}\nMax: {feature_data.max():.3f}"
    axes[row, col].text(0.98, 0.97, stats_text, transform=axes[row, col].transAxes,
                       fontsize=8, verticalalignment='top', horizontalalignment='right',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

# Remove the extra subplot
fig.delaxes(axes[1, 2])

plt.tight_layout()
plt.show()

In [ ]:
# Additional analysis: Diameter categories
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Asteroid Diameter Classification', fontsize=14, fontweight='bold')

# Create diameter categories
bins = [0, 1, 5, 10, 50, 100, 1000]
labels = ['<1 km', '1-5 km', '5-10 km', '10-50 km', '50-100 km', '>100 km']
df_diameter['diameter_category'] = pd.cut(df_diameter['diameter'], bins=bins, labels=labels, right=False)

# Count plot
category_counts = df_diameter['diameter_category'].value_counts().sort_index()
axes[0].bar(range(len(category_counts)), category_counts.values, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_xticks(range(len(category_counts)))
axes[0].set_xticklabels(category_counts.index, rotation=45, ha='right')
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Asteroid Count by Diameter Category', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, v in enumerate(category_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%',
            colors=['lightblue', 'lightgreen', 'lightyellow', 'lightcoral', 'lightpink', 'lightsalmon'],
            startangle=90)
axes[1].set_title('Proportion of Asteroids by Diameter Category', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("Asteroid Count by Diameter Category:")
print(category_counts)
print(f"\nTotal asteroids with diameter data: {len(df_diameter)}")

In [ ]:
# Summary statistics for the analysis
print("="*70)
print("EXPLORATORY DATA ANALYSIS SUMMARY")
print("="*70)
print(f"\nDataset Overview:")
print(f"  Total asteroids: {len(df_asteroid):,}")
print(f"  Asteroids with diameter data: {len(df_diameter):,} ({len(df_diameter)/len(df_asteroid)*100:.2f}%)")
print(f"  Complete records (all features): {len(plot_data):,} ({len(plot_data)/len(df_asteroid)*100:.2f}%)")

print(f"\nDiameter Statistics:")
print(f"  Mean: {df_diameter['diameter'].mean():.2f} km")
print(f"  Median: {df_diameter['diameter'].median():.2f} km")
print(f"  Std Dev: {df_diameter['diameter'].std():.2f} km")
print(f"  Range: {df_diameter['diameter'].min():.3f} - {df_diameter['diameter'].max():.2f} km")
print(f"  IQR: {df_diameter['diameter'].quantile(0.75) - df_diameter['diameter'].quantile(0.25):.2f} km")

print(f"\nTop Predictive Features (by correlation with diameter):")
for idx, (feature, corr_val) in enumerate(correlations.items(), 1):
    print(f"  {idx}. {feature:10s}: {corr_val:7.4f}")

print(f"\nKey Insights:")
print(f"  - Absolute Magnitude (H) shows strong negative correlation with diameter")
print(f"    (higher H means smaller asteroids)")
print(f"  - Albedo has limited predictive power for diameter")
print(f"  - Orbital parameters (a, e, i) show weak correlation with diameter")
print(f"  - The diameter distribution is right-skewed (most asteroids are small)")
print(f"  - Data quality issue: Only {len(df_diameter)/len(df_asteroid)*100:.2f}% have diameter measurements")
print("="*70)